# Binance Spot BTC quote-pair annual trading volume

This notebook collects monthly Binance Spot klines and aggregates annual trading activity for every symbol whose quote asset is BTC, regardless of its current status.

Three measures are retained:

- base-asset volume: quantity of the base asset traded;
- quote-asset volume: value traded in BTC;
- number of trades: count of trades reported by Binance.

Calendar years are defined in UTC. The current year is labeled `Year-to-date`. Pair-years before listing or after delisting are not filled with zero; they are absent when Binance returns no monthly kline.

In [ ]:
import json
import math
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

BASE_URL = "https://api.binance.com"
EXCHANGE_INFO_PATH = "/api/v3/exchangeInfo"
KLINES_PATH = "/api/v3/klines"
QUOTE_ASSET = "BTC"
KLINE_INTERVAL = "1M"
START_YEAR = 2017
REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES = 6
MAX_WORKERS = 20
KLINE_LIMIT = 1000
COLLECTION_TIME_UTC = pd.Timestamp.now(tz="UTC")
CURRENT_YEAR = COLLECTION_TIME_UTC.year
START_TIME_UTC = pd.Timestamp(f"{START_YEAR}-01-01", tz="UTC")
START_TIME_MS = int(START_TIME_UTC.timestamp() * 1_000)
END_TIME_MS = int(COLLECTION_TIME_UTC.timestamp() * 1_000)


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root by looking for AGENTS.md."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
OUTPUT_PATH = (
    REPOSITORY_ROOT
    / "data"
    / "exploration"
    / "binance_spot_btc_quote_pair_annual_volume.xlsx"
)
OUTPUT_PATH

In [ ]:
def request_json(
    path: str,
    parameters: dict[str, object] | None = None,
) -> object:
    """Request JSON from Binance with bounded transient-error retries."""
    query_string = urlencode(parameters or {})
    url = f"{BASE_URL}{path}"
    if query_string:
        url = f"{url}?{query_string}"

    for attempt in range(MAX_RETRIES + 1):
        request = Request(
            url,
            headers={"User-Agent": "tcc-crypto-research/0.1"},
        )
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return json.load(response)
        except HTTPError as error:
            is_transient = error.code == 429 or 500 <= error.code < 600
            if is_transient and attempt < MAX_RETRIES:
                retry_after = error.headers.get("Retry-After")
                delay_seconds = (
                    float(retry_after)
                    if retry_after is not None
                    else min(2**attempt, 30)
                )
                time.sleep(delay_seconds)
                continue
            response_body = error.read().decode("utf-8", errors="replace")
            raise RuntimeError(
                f"Binance returned HTTP {error.code} for {url}: {response_body}"
            ) from error
        except URLError as error:
            if attempt < MAX_RETRIES:
                time.sleep(min(2**attempt, 30))
                continue
            raise RuntimeError(f"Could not reach Binance at {url}: {error.reason}") from error

    raise RuntimeError(f"Request retry loop ended unexpectedly for {url}.")


def get_btc_quote_pairs() -> pd.DataFrame:
    """Return every Spot-enabled symbol that uses BTC as its quote asset."""
    payload = request_json(EXCHANGE_INFO_PATH)
    if not isinstance(payload, dict) or not isinstance(payload.get("symbols"), list):
        raise ValueError("Unexpected Binance exchangeInfo response.")

    records = [
        {
            "Trading Pair": item["symbol"],
            "Base Asset": item["baseAsset"],
            "Quote Asset": item["quoteAsset"],
            "Current Status": item["status"],
        }
        for item in payload["symbols"]
        if item.get("quoteAsset") == QUOTE_ASSET
        and item.get("isSpotTradingAllowed", False)
    ]
    if not records:
        raise ValueError(f"No Spot symbols quoted in {QUOTE_ASSET} were returned.")
    return pd.DataFrame(records).sort_values("Trading Pair").reset_index(drop=True)


MONTHLY_COLUMNS = [
    "Trading Pair",
    "Base Asset",
    "Quote Asset",
    "Current Status",
    "Month Open (UTC)",
    "Month Type",
    "Base Asset Volume",
    "Quote Asset Volume (BTC)",
    "Number of Trades",
    "Source URL",
]


def get_monthly_klines(pair: dict[str, str]) -> pd.DataFrame:
    """Return monthly volume fields for one symbol from 2017 onward."""
    payload = request_json(
        KLINES_PATH,
        {
            "symbol": pair["Trading Pair"],
            "interval": KLINE_INTERVAL,
            "startTime": START_TIME_MS,
            "endTime": END_TIME_MS,
            "limit": KLINE_LIMIT,
            "timeZone": 0,
        },
    )
    if not isinstance(payload, list):
        raise ValueError(f"Unexpected kline response for {pair['Trading Pair']}.")
    if len(payload) == KLINE_LIMIT:
        raise RuntimeError(
            f"{pair['Trading Pair']} reached the kline limit; pagination is required."
        )
    if not payload:
        return pd.DataFrame(columns=MONTHLY_COLUMNS)

    frame = pd.DataFrame(
        payload,
        columns=[
            "open_time_ms", "open", "high", "low", "close",
            "base_volume", "close_time_ms", "quote_volume",
            "number_of_trades", "taker_buy_base_volume",
            "taker_buy_quote_volume", "ignore",
        ],
    )
    frame["Month Open (UTC)"] = pd.to_datetime(
        frame["open_time_ms"], unit="ms", utc=True
    )
    frame["Base Asset Volume"] = pd.to_numeric(frame["base_volume"], errors="raise")
    frame["Quote Asset Volume (BTC)"] = pd.to_numeric(
        frame["quote_volume"], errors="raise"
    )
    frame["Number of Trades"] = pd.to_numeric(
        frame["number_of_trades"], errors="raise", downcast="integer"
    )
    current_month = COLLECTION_TIME_UTC.tz_localize(None).to_period("M")
    frame["Month Type"] = frame["Month Open (UTC)"].dt.tz_localize(None).dt.to_period("M").map(
        lambda period: "Partial month" if period == current_month else "Closed month"
    )
    frame["Trading Pair"] = pair["Trading Pair"]
    frame["Base Asset"] = pair["Base Asset"]
    frame["Quote Asset"] = pair["Quote Asset"]
    frame["Current Status"] = pair["Current Status"]
    frame["Source URL"] = f"{BASE_URL}{KLINES_PATH}"
    return frame[MONTHLY_COLUMNS]

In [ ]:
pairs = get_btc_quote_pairs()
print(f"Pairs returned: {len(pairs):,}")
pairs["Current Status"].value_counts(dropna=False)

In [ ]:
monthly_frames: list[pd.DataFrame] = []
errors: list[dict[str, str]] = []
pair_records = pairs.to_dict(orient="records")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(get_monthly_klines, pair): pair for pair in pair_records}
    for completed, future in enumerate(as_completed(futures), start=1):
        pair = futures[future]
        try:
            monthly_frames.append(future.result())
        except Exception as error:
            errors.append(
                {"Trading Pair": pair["Trading Pair"], "Error": str(error)}
            )
        if completed % 50 == 0 or completed == len(pair_records):
            print(f"Completed {completed:,}/{len(pair_records):,} symbols")

if errors:
    error_frame = pd.DataFrame(errors)
    display(error_frame)
    raise RuntimeError(
        f"Collection stopped because {len(errors)} symbol request(s) failed."
    )

monthly_volume = pd.concat(monthly_frames, ignore_index=True)
monthly_volume = monthly_volume.sort_values(
    ["Trading Pair", "Month Open (UTC)"]
).reset_index(drop=True)
print(f"Monthly observations: {len(monthly_volume):,}")
monthly_volume.head()

In [ ]:
monthly_volume["Year"] = monthly_volume["Month Open (UTC)"].dt.year
annual_volume = (
    monthly_volume.groupby(
        ["Trading Pair", "Base Asset", "Quote Asset", "Current Status", "Year"],
        as_index=False,
    )
    .agg(
        **{
            "Base Asset Volume": ("Base Asset Volume", "sum"),
            "Quote Asset Volume (BTC)": ("Quote Asset Volume (BTC)", "sum"),
            "Number of Trades": ("Number of Trades", "sum"),
            "Observed Months": ("Month Open (UTC)", "nunique"),
            "First Month Open (UTC)": ("Month Open (UTC)", "min"),
            "Last Month Open (UTC)": ("Month Open (UTC)", "max"),
        }
    )
)
annual_volume["Period Type"] = annual_volume["Year"].map(
    lambda year: "Year-to-date" if year == CURRENT_YEAR else "Historical year"
)
annual_volume["Source URL"] = f"{BASE_URL}{KLINES_PATH}"
annual_volume = annual_volume[
    [
        "Trading Pair", "Base Asset", "Quote Asset", "Current Status",
        "Year", "Period Type", "Base Asset Volume",
        "Quote Asset Volume (BTC)", "Number of Trades",
        "Observed Months", "First Month Open (UTC)",
        "Last Month Open (UTC)", "Source URL",
    ]
].sort_values(["Trading Pair", "Year"]).reset_index(drop=True)

annual_volume.head(12)

In [ ]:
def totals_match(left: float, right: float) -> bool:
    """Compare floating-point totals using a scale-aware tolerance."""
    return math.isclose(float(left), float(right), rel_tol=1e-12, abs_tol=1e-8)


duplicate_months = int(
    monthly_volume.duplicated(["Trading Pair", "Month Open (UTC)"]).sum()
)
negative_volume_rows = int(
    (
        (monthly_volume["Base Asset Volume"] < 0)
        | (monthly_volume["Quote Asset Volume (BTC)"] < 0)
        | (monthly_volume["Number of Trades"] < 0)
    ).sum()
)
raw_base_total = monthly_volume["Base Asset Volume"].sum()
annual_base_total = annual_volume["Base Asset Volume"].sum()
raw_quote_total = monthly_volume["Quote Asset Volume (BTC)"].sum()
annual_quote_total = annual_volume["Quote Asset Volume (BTC)"].sum()
raw_trade_total = int(monthly_volume["Number of Trades"].sum())
annual_trade_total = int(annual_volume["Number of Trades"].sum())
base_volume_tolerance = max(1e-8, abs(float(raw_base_total)) * 1e-12)
quote_volume_tolerance = max(1e-8, abs(float(raw_quote_total)) * 1e-12)
current_year_label_errors = int(
    (
        annual_volume.loc[annual_volume["Year"] == CURRENT_YEAR, "Period Type"]
        != "Year-to-date"
    ).sum()
)

validation = pd.DataFrame(
    [
        ("Duplicate pair-month rows", duplicate_months, 0, duplicate_months, 0, duplicate_months == 0, "Each pair-month must be unique."),
        ("Negative activity rows", negative_volume_rows, 0, negative_volume_rows, 0, negative_volume_rows == 0, "Volumes and trade counts cannot be negative."),
        ("Base-volume annual tie-out", annual_base_total, raw_base_total, annual_base_total - raw_base_total, base_volume_tolerance, totals_match(annual_base_total, raw_base_total), "Annual sums must reconcile to monthly source rows within the displayed floating-point tolerance."),
        ("Quote-volume annual tie-out", annual_quote_total, raw_quote_total, annual_quote_total - raw_quote_total, quote_volume_tolerance, totals_match(annual_quote_total, raw_quote_total), "Annual BTC sums must reconcile to monthly source rows within the displayed floating-point tolerance."),
        ("Trade-count annual tie-out", annual_trade_total, raw_trade_total, annual_trade_total - raw_trade_total, 0, annual_trade_total == raw_trade_total, "Annual trade counts must reconcile exactly."),
        ("Current-year YTD labels", current_year_label_errors, 0, current_year_label_errors, 0, current_year_label_errors == 0, f"All {CURRENT_YEAR} rows must be labeled Year-to-date."),
    ],
    columns=["Check", "Actual", "Expected", "Difference", "Tolerance", "Passed", "Notes"],
)
validation["Status"] = validation["Passed"].map({True: "OK", False: "FAIL"})
validation = validation[["Check", "Actual", "Expected", "Difference", "Tolerance", "Status", "Notes"]]

if not (validation["Status"] == "OK").all():
    display(validation)
    raise ValueError("Validation failed; the workbook will not be generated.")

validation

In [ ]:
def style_header(worksheet, row_number: int, fill: PatternFill, font: Font) -> None:
    """Apply a consistent style to a worksheet header row."""
    for cell in worksheet[row_number]:
        if cell.value is not None:
            cell.fill = fill
            cell.font = font
            cell.alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )


def apply_workbook_style(
    writer: pd.ExcelWriter,
    annual_rows: int,
    monthly_rows: int,
    validation_rows: int,
    methodology_rows: int,
) -> None:
    """Apply clean, data-focused formatting to the generated workbook."""
    header_fill = PatternFill("solid", fgColor="17365D")
    header_font = Font(color="FFFFFF", bold=True)
    label_fill = PatternFill("solid", fgColor="F3F6F9")
    section_fill = PatternFill("solid", fgColor="D9EAF7")
    ytd_fill = PatternFill("solid", fgColor="FFF2CC")
    ok_fill = PatternFill("solid", fgColor="E2F0D9")
    fail_fill = PatternFill("solid", fgColor="F4CCCC")
    thin_gray = Side(style="thin", color="E5E7EB")

    annual_sheet = writer.sheets["Annual Volume"]
    annual_sheet.sheet_view.showGridLines = False
    annual_sheet.freeze_panes = "E2"
    annual_sheet.auto_filter.ref = annual_sheet.dimensions
    annual_sheet.row_dimensions[1].height = 36
    style_header(annual_sheet, 1, header_fill, header_font)
    annual_widths = {
        "A": 16, "B": 12, "C": 12, "D": 14, "E": 9,
        "F": 16, "G": 22, "H": 25, "I": 18, "J": 17,
        "K": 24, "L": 24, "M": 48,
    }
    for column, width in annual_widths.items():
        annual_sheet.column_dimensions[column].width = width
    for row in annual_sheet.iter_rows(min_row=2, max_row=annual_rows + 1):
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="center")
        row[3].alignment = Alignment(horizontal="center", vertical="center")
        row[4].alignment = Alignment(horizontal="center", vertical="center")
        row[5].alignment = Alignment(horizontal="center", vertical="center")
        row[6].number_format = "#,##0.0000"
        row[7].number_format = "#,##0.00000000"
        row[8].number_format = "#,##0"
        row[9].number_format = "0"
        row[10].number_format = "yyyy-mm-dd hh:mm:ss"
        row[11].number_format = "yyyy-mm-dd hh:mm:ss"
        if row[5].value == "Year-to-date":
            for cell in row:
                cell.fill = ytd_fill
        row[12].hyperlink = row[12].value
        row[12].style = "Hyperlink"

    monthly_sheet = writer.sheets["Monthly Data"]
    monthly_sheet.sheet_view.showGridLines = False
    monthly_sheet.freeze_panes = "E2"
    monthly_sheet.auto_filter.ref = monthly_sheet.dimensions
    monthly_sheet.row_dimensions[1].height = 36
    style_header(monthly_sheet, 1, header_fill, header_font)
    monthly_widths = {
        "A": 16, "B": 12, "C": 12, "D": 14, "E": 24,
        "F": 15, "G": 22, "H": 25, "I": 18, "J": 48,
    }
    for column, width in monthly_widths.items():
        monthly_sheet.column_dimensions[column].width = width
    for row in monthly_sheet.iter_rows(min_row=2, max_row=monthly_rows + 1):
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="center")
        row[3].alignment = Alignment(horizontal="center", vertical="center")
        row[4].number_format = "yyyy-mm-dd hh:mm:ss"
        row[4].alignment = Alignment(horizontal="center", vertical="center")
        row[5].alignment = Alignment(horizontal="center", vertical="center")
        row[6].number_format = "#,##0.0000"
        row[7].number_format = "#,##0.00000000"
        row[8].number_format = "#,##0"
        row[9].hyperlink = row[9].value
        row[9].style = "Hyperlink"

    validation_sheet = writer.sheets["Validation"]
    validation_sheet.sheet_view.showGridLines = False
    validation_sheet.freeze_panes = "A2"
    validation_sheet.row_dimensions[1].height = 30
    style_header(validation_sheet, 1, header_fill, header_font)
    for column, width in {"A": 31, "B": 22, "C": 22, "D": 22, "E": 18, "F": 11, "G": 58}.items():
        validation_sheet.column_dimensions[column].width = width
    for row in validation_sheet.iter_rows(min_row=2, max_row=validation_rows + 1):
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
        if row[0].value in {"Duplicate pair-month rows", "Negative activity rows", "Trade-count annual tie-out", "Current-year YTD labels"}:
            for cell in row[1:5]:
                cell.number_format = "#,##0"
        elif row[0].value == "Base-volume annual tie-out":
            for cell in row[1:5]:
                cell.number_format = "#,##0.0000"
        elif row[0].value == "Quote-volume annual tie-out":
            for cell in row[1:5]:
                cell.number_format = "#,##0.00000000"
        row[5].fill = ok_fill if row[5].value == "OK" else fail_fill
        row[5].font = Font(bold=True)

    methodology_sheet = writer.sheets["Methodology"]
    methodology_sheet.sheet_view.showGridLines = False
    methodology_sheet.freeze_panes = "A4"
    methodology_sheet.merge_cells("A1:B1")
    methodology_sheet["A1"] = "Binance Spot BTC Quote-Pair Annual Trading Volume"
    methodology_sheet["A1"].fill = header_fill
    methodology_sheet["A1"].font = Font(color="FFFFFF", bold=True, size=15)
    methodology_sheet["A1"].alignment = Alignment(vertical="center")
    methodology_sheet.row_dimensions[1].height = 30
    methodology_sheet.column_dimensions["A"].width = 34
    methodology_sheet.column_dimensions["B"].width = 105
    style_header(methodology_sheet, 3, section_fill, Font(color="17365D", bold=True))
    for row in methodology_sheet.iter_rows(min_row=4, max_row=methodology_rows + 3):
        row[0].fill = label_fill
        row[0].font = Font(color="334155", bold=True)
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="top", wrap_text=True)


def export_annual_volume_workbook(
    annual_data: pd.DataFrame,
    monthly_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    output_path: Path,
) -> Path:
    """Export annual results, monthly source rows, checks, and methodology."""
    annual_export = annual_data.copy()
    monthly_export = monthly_data.drop(columns="Year").copy()
    for column in ["First Month Open (UTC)", "Last Month Open (UTC)"]:
        annual_export[column] = annual_export[column].dt.tz_localize(None)
    monthly_export["Month Open (UTC)"] = monthly_export["Month Open (UTC)"].dt.tz_localize(None)

    methodology = pd.DataFrame(
        [
            ("Collection timestamp (UTC)", COLLECTION_TIME_UTC.strftime("%Y-%m-%d %H:%M:%S")),
            ("Universe definition", "All Binance Spot symbols with BTC as quoteAsset and Spot trading allowed by exchangeInfo, regardless of current status."),
            ("Current-status treatment", "Current status is descriptive only; no status is excluded. Historical investability is not inferred from current status."),
            ("Annual period", "Calendar year in UTC. The current year is labeled Year-to-date."),
            ("Source interval", "Monthly Spot klines (1M). Annual measures are sums of monthly fields."),
            ("Base Asset Volume", "Sum of kline field 5: quantity of the pair's base asset traded. Units differ across pairs."),
            ("Quote Asset Volume (BTC)", "Sum of kline field 7: trading value denominated in BTC, comparable across the selected pairs."),
            ("Number of Trades", "Sum of kline field 8: number of trades reported by Binance."),
            ("Observed Months", "Number of monthly klines contributing to the pair-year. Fewer than 12 may reflect listing, delisting, missing activity, or the partial current year."),
            ("Missing-year treatment", "Pair-years without a returned monthly kline are absent, not imputed as zero."),
            ("Current-month treatment", "The open current-month kline is included using values available at the collection timestamp and labeled Partial month in Monthly Data."),
            ("Numeric representation", "Binance decimal strings are converted to pandas numeric values and stored as Excel numbers."),
            ("Pairs returned", len(pairs)),
            ("Annual rows", len(annual_export)),
            ("Monthly source rows", len(monthly_export)),
            ("Exchange information endpoint", f"{BASE_URL}{EXCHANGE_INFO_PATH}"),
            ("Kline endpoint", f"{BASE_URL}{KLINES_PATH}"),
            ("Official documentation", "https://developers.binance.com/en/docs/binance-spot-api-docs/rest-api/market-data-endpoints#klinecandlestick-data"),
        ],
        columns=["Field", "Value"],
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(
        output_path, engine="openpyxl", datetime_format="yyyy-mm-dd hh:mm:ss"
    ) as writer:
        annual_export.to_excel(writer, sheet_name="Annual Volume", index=False)
        monthly_export.to_excel(writer, sheet_name="Monthly Data", index=False)
        validation_data.to_excel(writer, sheet_name="Validation", index=False)
        methodology.to_excel(writer, sheet_name="Methodology", index=False, startrow=2)
        apply_workbook_style(
            writer, len(annual_export), len(monthly_export),
            len(validation_data), len(methodology),
        )
    return output_path

In [ ]:
generated_path = export_annual_volume_workbook(
    annual_volume, monthly_volume, validation, OUTPUT_PATH
)
print(f"Workbook created: {generated_path}")

## Interpretation notes

`Quote Asset Volume (BTC)` is the directly comparable value measure because every selected pair is quoted in BTC. `Base Asset Volume` remains useful within a pair but should not be compared directly across different assets without a unit conversion.

The annual table contains observed pair-years only. An absent row does not mean zero trading volume. The current `exchangeInfo` universe and status fields also do not reconstruct point-in-time historical eligibility; that requires a separate universe-selection protocol.